# Final TESS Transit Classification — Optimized for Extreme Class Imbalance

This notebook converts and expands the provided Python script into a fully documented, didactic, and **step-by-step** workflow.
We train a 1D CNN to classify TESS light curves into *transit* (planet candidate) vs *non-transit* under **severe class imbalance**.

**Key strategies covered:**

- **Balanced augmentation** (equal samples per class) to mitigate imbalance during training.  
- **Focal loss** (tunable `gamma` and `alpha`) to emphasize hard examples and rare positives.  
- **Threshold optimization** using **Youden’s J** from the ROC curve (don’t use the default 0.5).  
- **Simplified CNN architecture** to reduce overfitting.  
- **AUC-centric monitoring** with early stopping and LR scheduling.

> **What you’ll learn**
>
> 1. Why balanced training batches help under extreme imbalance.  
> 2. How focal loss reshapes the gradient to focus on hard/rare samples.  
> 3. How to pick a **data-driven** decision threshold that best trades off TPR/FPR.  
> 4. How to evaluate with AUC-ROC rather than accuracy (which can be misleading).  
> 5. How to visualize confusion matrices and sample light curves with predictions.


## 1. Prerequisites & Data

**Dependencies** (install if needed):

```bash
pip install numpy pandas scikit-learn matplotlib tensorflow
```

> We intentionally avoid additional plotting libraries to keep dependencies compact.  
> If you already have a working scientific Python/TensorFlow stack, you can skip installations.

**Expected dataset**: a CSV file named **`tess_data.csv`** in the working directory with:

- **Light-curve samples**: `flux_0000, flux_0001, ..., flux_0999` (or up to `n_bins-1`)  
- **Flux uncertainties**: `flux_err_0000, ..., flux_err_0999`  
- **Label**: `label` (0 = Non-Planet, 1 = Planet)  
- **Metadata** (used for plots/titles): `toi_name, tic, disp, period_d, t0_bjd, dur_hr, sector`

You can change the filename or number of bins via parameters in the **Data Loading** section.


## 2. Environment Setup

Set up imports, suppress noisy warnings, and fix seeds for reproducibility.  
(Exact reproducibility on GPUs may still vary across hardware/driver versions.)


In [5]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import IPython
    working_directory = "/".join(
            IPython.extract_module_locals()[1]["__vsc_ipynb_file__"].split("/")[:-1]
        )
    print("Setting working directory to: ", working_directory)
    print(os.chdir(working_directory))
except Exception as e:
    print("It was impossible to set your directory as the current one because of the following message")
    print(e)
    print("The working directory is: ", os.getcwd())

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("="*70)
print("FINAL TESS CLASSIFICATION")
print("="*70)

# Optional: make TF less eager to pre-allocate all GPU memory (if using GPU)
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    if gpus:
        print(f"Enabled memory growth for {len(gpus)} GPU(s).")
except Exception as e:
    print("GPU setup note:", e)

print("TF version:", tf.__version__)


Setting working directory to:  /ca25/ext_volume/group9_/assignment2
None
FINAL TESS CLASSIFICATION
TF version: 2.21.0


## 3. Focal Loss (for severe imbalance)

**Why focal loss?** With extreme imbalance, the model can get “lazy”—it learns to do well by focusing on the majority class.  
Focal loss down-weights *easy* examples and concentrates gradient on *hard* ones by adding a modulating factor \((1 - p_t)^\gamma\).  
We also use class weighting via \(\alpha\) to up-weight the rare positive class.

- **`gamma`** (focusing parameter): higher values put more emphasis on hard examples.  
- **`alpha`** (class weight): weight for positive class (1); negative class gets \(1 - \alpha\).


In [6]:
def focal_loss(gamma=2.5, alpha=0.75):
    """Focal loss optimized for severe imbalance (binary)."""
    def focal_loss_fixed(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
        
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_factor = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        focal_weight = alpha_factor * K.pow(1 - pt, gamma)
        bce = -K.log(pt)
        return K.mean(focal_weight * bce)
    return focal_loss_fixed


## 4. Balanced Augmentation

We **balance the training set** to a fixed number of samples per class.  
If a class has too few samples, we create augmented variants (noise, scale, shift, combo).  
This prevents the model from being swamped by the majority class during training.


In [7]:
def create_balanced_dataset(X, y, samples_per_class=400):
    """Create a perfectly balanced dataset via lightweight augmentations."""
    print("\n" + "="*70)
    print("CREATING BALANCED DATASET")
    print("="*70)
    
    X_class0 = X[y == 0]
    X_class1 = X[y == 1]
    
    print(f"Original - Class 0: {len(X_class0)}, Class 1: {len(X_class1)}")
    
    def augment_to_target(X_orig, n_target):
        if len(X_orig) >= n_target:
            idx = np.random.choice(len(X_orig), n_target, replace=False)
            return X_orig[idx]
        
        X_result = [X_orig]
        while len(np.vstack(X_result)) < n_target:
            # number we still need (cap to avoid oversampling too big chunks)
            n_needed = n_target - len(np.vstack(X_result))
            idx = np.random.choice(len(X_orig), min(len(X_orig), n_needed))
            
            aug_type = np.random.rand()
            if aug_type < 0.25:
                # Additive Gaussian noise
                X_aug = X_orig[idx] + np.random.normal(0, 0.01, (len(idx), X_orig.shape[1]))
            elif aug_type < 0.5:
                # Multiplicative scaling
                scale = 1.0 + np.random.uniform(-0.03, 0.03, (len(idx), 1))
                X_aug = X_orig[idx] * scale
            elif aug_type < 0.75:
                # Circular shift (time shift)
                shifts = np.random.randint(-20, 20, len(idx))
                X_aug = np.array([np.roll(X_orig[i], s) for i, s in zip(idx, shifts)])
            else:
                # Mild combo: small scale + small noise
                X_aug = X_orig[idx] * (1.0 + np.random.uniform(-0.02, 0.02, (len(idx), 1)))
                X_aug += np.random.normal(0, 0.008, X_aug.shape)
            
            X_result.append(X_aug)
        
        X_final = np.vstack(X_result)
        return X_final[:n_target]
    
    X0_bal = augment_to_target(X_class0, samples_per_class)
    X1_bal = augment_to_target(X_class1, samples_per_class)
    
    print(f"Balanced - Class 0: {len(X0_bal)}, Class 1: {len(X1_bal)}")
    
    X_balanced = np.vstack([X0_bal, X1_bal])
    y_balanced = np.concatenate([np.zeros(samples_per_class), np.ones(samples_per_class)])
    
    # Shuffle
    idx = np.arange(len(X_balanced))
    np.random.shuffle(idx)
    
    return X_balanced[idx], y_balanced[idx]


## 5. Data Loading, Splitting & Standardization

We split **before** augmentation (to avoid leakage), then **balance only the training split**.  
We standardize the flux (zero mean / unit variance) using statistics from the training set only.

**Notes**

- Error bars `X_err` are **not** standardized (kept in their original scale).  
- We keep the **test metadata** to produce nicer titles in the sample light-curve plots.


In [8]:
def load_data(csv_path='tess_data.csv', n_bins=1000):
    """Load CSV, split, balance train set, and standardize features."""
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)
    
    df = pd.read_csv(csv_path)
    print(f"Dataset: {df.shape[0]} samples")
    
    flux_cols = [f'flux_{i:04d}' for i in range(n_bins)]
    flux_err_cols = [f'flux_err_{i:04d}' for i in range(n_bins)]
    X = df[flux_cols].values
    X_err = df[flux_err_cols].values
    y = df['label'].values
    
    metadata_cols = ['toi_name', 'tic', 'label', 'disp', 'period_d', 't0_bjd', 'dur_hr', 'sector']
    metadata = df[metadata_cols]
    
    print("\nOriginal distribution:")
    print(f"  Class 0: {(y==0).sum()}, Class 1: {(y==1).sum()}")
    if (y==0).sum() > 0:
        print(f"  Ratio: {(y==1).sum() / (y==0).sum():.2f}:1")
    
    # Train/test split (keep errors aligned; stratify to preserve class ratio)
    X_train, X_test, y_train, y_test, X_err_train, X_err_test, idx_train, idx_test = train_test_split(
        X, y, X_err, np.arange(len(y)),
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    print(f"\nInitial split - Train: {len(X_train)}, Test: {len(X_test)}")
    
    # Balance training set
    X_train, y_train = create_balanced_dataset(X_train, y_train, samples_per_class=350)
    
    # Standardize (fit on train, apply to test)
    print("\n" + "="*70)
    print("STANDARDIZATION")
    print("="*70)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    print(f"Train: mean={X_train.mean():.6f}, std={X_train.std():.6f}")
    print(f"Test:  mean={X_test.mean():.6f}, std={X_test.std():.6f}")
    
    # Reshape for Conv1D: (samples, timesteps, channels)
    X_train = X_train.reshape(-1, n_bins, 1)
    X_test = X_test.reshape(-1, n_bins, 1)
    
    metadata_test = metadata.iloc[idx_test].reset_index(drop=True)
    
    print(f"\nFinal - X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"Train dist: 0={( y_train==0).sum()}, 1={(y_train==1).sum()}")
    
    # Return standardized test for model input, but also return the standardized
    # copy (X_test_orig) so we can inverse-transform for plotting with error bars.
    return X_train, X_test, y_train, y_test, metadata_test, X_test.copy(), X_err_test, scaler


## 6. CNN Architectures A Simpler 1D CNN (to curb overfitting)
CNN add by the prof and then 3 by us

(A Simpler 1D CNN to curb overfitting)A compact ConvNet with **BatchNorm**, **Dropout**, and **Global Average Pooling** is often enough for
noisy, small-ish 1D signals. We also add mild L2 on the dense layers. The goal is a strong baseline
that generalizes well, not a gigantic model that memorizes. 


## 6.1 CNN 1 - Original Cinv1D

In [9]:
def build_simple_cnn(n_bins=1000):
    """Simpler CNN to prevent overfitting on small datasets."""
    print("\n" + "="*70)
    print("BUILDING SIMPLIFIED CNN")
    print("="*70)
    
    model = models.Sequential([
        layers.Input(shape=(n_bins, 1)),
        
        # Feature extraction
        layers.Conv1D(64, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),
        
        layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),
        
        layers.Conv1D(256, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.4),
        
        # Classification head
        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.2),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss=focal_loss(gamma=2.5, alpha=0.75),
        metrics=['accuracy',
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall'),
                 keras.metrics.AUC(name='auc')]
    )
    
    model.summary()
    print("\nUsing Focal Loss (gamma=2.5, alpha=0.75)")
    return model


## 6.2 CNN 2 — Conv2D

In [10]:
def build_cnn2_2d(loss_type="focal"):

    # Scegliamo la funzione di perdita
    if loss_type == "focal":
        selected_loss = focal_loss(gamma=2.5, alpha=0.75)

    elif loss_type == "bce":
        selected_loss = "binary_crossentropy"

    else:
        raise ValueError("Usa 'focal' oppure 'bce'")

    # Architettura bidimensionale
    model = models.Sequential([
        layers.Input(shape=(50, 20, 1)),

        # Primo blocco convoluzionale
        layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.3),

        # Secondo blocco convoluzionale
        layers.Conv2D(filters=128, kernel_size=(3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.3),

        layers.Conv2D(filters=256, kernel_size=(3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        # Riduzione delle feature maps
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),

        # Classificatore
        layers.Dense(512, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Dropout(0.2),
        layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.Dropout(0.2),        
        # Uscita binaria
        layers.Dense(1, activation="sigmoid")

    ], name="cnn2_2d")

    # Configurazione del training
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss=selected_loss,
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

## 6.3 CNN 3 — Modified Conv1D

In [52]:

# CNN 3 — Deep 1D Architecture

def build_cnn3_1d(n_bins=1000):

    from tensorflow.keras import layers, models, regularizers

    inputs = layers.Input(shape=(n_bins, 1))
    x = inputs

    conv_config = [
        (32, 7),
        (64, 5),
        (128, 5),
        (128, 3),
        (256, 3)
    ]

    for i, (filters, kernel_size) in enumerate(conv_config):

        x = layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            padding="same",
            kernel_regularizer=regularizers.l2(1e-4)
        )(x)

        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        if i < 3:
            x = layers.MaxPooling1D(pool_size=2)(x)

        if i == 2:
            x = layers.Dropout(0.20)(x)

        if i == 4:
            x = layers.Dropout(0.25)(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.20)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CNN3_Deep_1D"
    )



## 6.4 CNN 4 — Modified Conv2D

In [53]:

# CNN 4 — Compact 2D Architecture

def build_cnn4_2d(input_shape=(50, 20, 1)):

    from tensorflow.keras import layers, models, regularizers

    inputs = layers.Input(shape=input_shape)

    # Convolutional block 1
    x = layers.Conv2D(
        64,
        kernel_size=(3, 3),
        padding="same",
        kernel_regularizer=regularizers.l2(1e-4)
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Convolutional block 2
    x = layers.Conv2D(
        128,
        kernel_size=(3, 3),
        padding="same",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Dropout(0.25)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.20)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CNN4_Compact_2D"
    )





## 7. Training with AUC Monitoring, Early Stopping & LR Scheduling

We monitor **validation AUC** (not accuracy) and:

- **EarlyStopping** on `val_auc` with patience to stop when progress stalls.  
- **ReduceLROnPlateau** to gently lower the LR when AUC plateaus.  
- **ModelCheckpoint** to persist the best model by AUC.

> Tip: If your dataset is *very* small, increase dropout and/or reduce dense layers further.


In [11]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=100):
    """Train the model with AUC-centric callbacks."""
    print("\n" + "="*70)
    print("TRAINING")
    print("="*70)
    
    callbacks = [
        EarlyStopping(
            monitor='val_auc',
            patience=20,
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_auc',
            factor=0.5,
            patience=8,
            min_lr=1e-7,
            mode='max',
            verbose=1
        ),
        ModelCheckpoint(
            'best_model_final.keras',
            monitor='val_auc',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    return history


## 8. Evaluation with **Optimal Threshold** (don’t default to 0.5)

The default threshold (0.5) is rarely optimal with imbalanced data.  
We compute ROC, then choose the threshold that maximizes **Youden’s J** (\(\mathrm{TPR} - \mathrm{FPR}\)).
We report both the default and the optimal settings.


In [12]:
def evaluate_with_optimal_threshold(model, X_test, y_test):
    """Find an optimal threshold from ROC (Youden's J) and evaluate."""
    print("\n" + "="*70)
    print("THRESHOLD OPTIMIZATION & EVALUATION")
    print("="*70)
    
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    
    # Youden's J statistic
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    
    print(f"\nOptimal threshold: {optimal_threshold:.4f} (default=0.5)")
    print(f"  At this threshold: TPR={tpr[optimal_idx]:.4f}, FPR={fpr[optimal_idx]:.4f}")
    
    # Predictions with optimal vs default thresholds
    y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)
    y_pred_default = (y_pred_proba >= 0.5).astype(int)
    
    # Metrics
    acc_optimal = accuracy_score(y_test, y_pred_optimal)
    acc_default = accuracy_score(y_test, y_pred_default)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print("\nResults:")
    print(f"  AUC-ROC: {auc:.4f}")
    print(f"  Accuracy (default threshold=0.5): {acc_default:.4f} ({acc_default*100:.2f}%)")
    print(f"  Accuracy (optimal threshold={optimal_threshold:.4f}): {acc_optimal:.4f} ({acc_optimal*100:.2f}%)")
    
    print("\nWith optimal threshold:")
    print(classification_report(y_test, y_pred_optimal,
                                target_names=['Non-Planet', 'Planet'],
                                digits=4,
                                zero_division=0))
    
    print("\nPrediction distribution (optimal threshold):")
    print(f"  Predicted 0: {(y_pred_optimal == 0).sum()}")
    print(f"  Predicted 1: {(y_pred_optimal == 1).sum()}")
    print("True distribution:")
    print(f"  True 0: {(y_test == 0).sum()}")
    print(f"  True 1: {(y_test == 1).sum()}")
    
    return y_pred_optimal, y_pred_proba, optimal_threshold


## 9. Visualization (Matplotlib-only)

We save:
- **Confusion matrix** (`confusion_matrix_final.png`) with counts and percentages.  
- **Training curves** (`training_history_final.png`).  
- **Sample light curves with predictions** (`sample_lightcurves_predictions.png`).

> We use **Matplotlib** exclusively to minimize dependencies.


In [ ]:
def plot_lightcurves_with_predictions(X_test_orig, X_err_test, y_test, y_pred, y_pred_proba, 
                                       metadata_test, scaler, threshold, n_samples=6,
                                       save_path='sample_lightcurves_predictions.png'):
    """Plot light curves with error bars and prediction info; save to file."""
    print("\n" + "="*70)
    print(f"PLOTTING LIGHTCURVES WITH PREDICTIONS (n={n_samples})")
    print("="*70)
    
    n_samples = min(n_samples, len(X_test_orig))
    
    # Select diverse samples: correct/incorrect for both classes
    correct_planet = np.where((y_test == 1) & (y_pred == 1))[0]
    incorrect_planet = np.where((y_test == 1) & (y_pred == 0))[0]
    correct_nonplanet = np.where((y_test == 0) & (y_pred == 0))[0]
    incorrect_nonplanet = np.where((y_test == 0) & (y_pred == 1))[0]
    
    selected_idx = []
    per_category = max(1, n_samples // 4)
    
    for idx_list in [correct_planet, incorrect_planet, correct_nonplanet, incorrect_nonplanet]:
        if len(idx_list) > 0:
            n_select = min(per_category, len(idx_list))
            selected_idx.extend(np.random.choice(idx_list, n_select, replace=False))
    
    while len(selected_idx) < n_samples:
        remaining = list(set(range(len(y_test))) - set(selected_idx))
        if remaining:
            selected_idx.append(np.random.choice(remaining))
        else:
            break
    
    selected_idx = np.array(selected_idx[:n_samples])
    
    # Figure layout
    n_cols = 2
    n_rows = (n_samples + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    if n_samples == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for plot_i, idx in enumerate(selected_idx):
        ax = axes[plot_i]
        
        # Inverse transform to original scale for plotting
        flux_norm = X_test_orig[idx].flatten()
        flux_err = X_err_test[idx]
        flux_original = scaler.inverse_transform(flux_norm.reshape(1, -1)).flatten()
        
        time_bins = np.arange(len(flux_original))
        
        # Metadata
        toi_name = metadata_test.loc[idx, 'toi_name']
        tic = metadata_test.loc[idx, 'tic']
        disp = metadata_test.loc[idx, 'disp']
        sector = metadata_test.loc[idx, 'sector']
        
        true_label = y_test[idx]
        pred_label = y_pred[idx]
        pred_prob = y_pred_proba[idx]
        
        is_correct = (true_label == pred_label)
        true_str = 'Transit' if true_label == 1 else 'Non-Transit'
        pred_str = 'Transit' if pred_label == 1 else 'Non-Transit'
        
        # Errorbar plot
        ax.errorbar(time_bins, flux_original, yerr=flux_err, fmt='o', markersize=2,
                    ecolor='gray', elinewidth=0.5, capsize=0, alpha=0.6, label='Data')
        
        # Baseline median
        baseline = np.median(flux_original)
        ax.axhline(baseline, linestyle='--', linewidth=1, alpha=0.7, label='Baseline')
        
        ax.set_xlabel('Time Bin', fontsize=10, fontweight='bold')
        ax.set_ylabel('Flux (original scale)', fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right', fontsize=8)
        
        status_symbol = '✓' if is_correct else '✗'
        color = 'green' if is_correct else 'red'
        title = (f'TOI {toi_name} (TIC {tic}, {disp}) - TESS Sector {sector}\n'
                 f'True: {true_str} | Pred: {pred_str} (p={pred_prob:.3f}) {status_symbol}')
        ax.set_title(title, fontsize=10, fontweight='bold', color=color, pad=10)
        
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2.0)
    
    # Hide unused axes
    for j in range(n_samples, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle(f'Sample Light-curve Predictions (Threshold={threshold:.3f})',
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    plt.close()


def plot_all(y_test, y_pred, y_pred_proba, history, metadata_test, X_test, threshold,
             X_test_orig=None, X_err_test=None, scaler=None):
    """Create and save confusion matrix and training curves. Optionally plot light curves."""
    print("\n" + "="*70)
    print("VISUALIZATIONS")
    print("="*70)
    
    # Confusion matrix (Matplotlib-only)
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(2),
           yticks=np.arange(2),
           xticklabels=['Non-Planet', 'Planet'],
           yticklabels=['Non-Planet', 'Planet'],
           xlabel='Predicted', ylabel='True',
           title=f'Confusion Matrix (threshold={threshold:.3f})')
    
    # Add counts and percentages
    total = cm.sum()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            count = cm[i, j]
            pct = (count / total * 100) if total > 0 else 0.0
            ax.text(j, i, f"{count}\n({pct:.1f}%)", ha='center', va='center', color='black', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('confusion_matrix_final.png', dpi=300)
    print("Saved: confusion_matrix_final.png")
    plt.close()
    
    # Training history
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'),
               ('auc', 'AUC'), ('recall', 'Recall')]
    
    for idx, (metric, title) in enumerate(metrics):
        ax = axes[idx // 2, idx % 2]
        if metric in history.history and f'val_{metric}' in history.history:
            ax.plot(history.history[metric], label='Train', linewidth=2)
            ax.plot(history.history[f'val_{metric}'], label='Val', linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(title)
            ax.set_title(f'{title} vs Epoch', fontweight='bold')
            ax.legend()
            ax.grid(alpha=0.3)
    
    plt.suptitle('Training History - Final Model', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_history_final.png', dpi=300)
    print("Saved: training_history_final.png")
    plt.close()
    
    # Optional: light-curve panel
    if X_test_orig is not None and X_err_test is not None and scaler is not None:
        plot_lightcurves_with_predictions(X_test_orig, X_err_test, y_test, y_pred, 
                                          y_pred_proba, metadata_test, scaler, threshold, n_samples=6)

#add by me for saving the results of the experiments

def save_experiment_results(
    experiment_name,
    model,
    y_true,
    y_pred,
    y_prob,
    threshold,
    history=None
):
    """Save model, predictions, metrics and plots."""

    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    from sklearn.metrics import (
        confusion_matrix,
        ConfusionMatrixDisplay,
        classification_report,
        roc_curve,
        roc_auc_score,
        accuracy_score,
        precision_score,
        recall_score,
        f1_score
    )

    # 1. Create output directory
    output_dir = Path("CNN_outputs") / experiment_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # 2. Save trained model
    model.save(output_dir / "best_model.keras")

    # 3. Save predictions
    np.savez(
        output_dir / "predictions.npz",
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob,
        threshold=threshold
    )

    # 4. Calculate and save metrics
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    auc = roc_auc_score(y_true, y_prob)

    default_pred = (np.asarray(y_prob) >= 0.5).astype(int)

    report = f"""Experiment: {experiment_name}

Parameters: {model.count_params()}
Optimal threshold: {threshold:.6f}

ROC-AUC: {auc:.6f}
Accuracy (threshold=0.5): {accuracy_score(y_true, default_pred):.6f}
Accuracy (optimal): {accuracy_score(y_true, y_pred):.6f}
Precision (Planet): {precision_score(y_true, y_pred):.6f}
Recall (Planet): {recall_score(y_true, y_pred):.6f}
F1-score (Planet): {f1_score(y_true, y_pred):.6f}

Confusion matrix:
{cm}

Classification report:
{classification_report(
    y_true,
    y_pred,
    target_names=["Non-Planet", "Planet"]
)}
"""

    (output_dir / "results.txt").write_text(
        report, encoding="utf-8"
    )

    # 5. Save confusion matrix
    fig, ax = plt.subplots(figsize=(7, 6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-Planet", "Planet"]
    )

    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"{experiment_name} - Confusion Matrix")

    fig.savefig(
        output_dir / "confusion_matrix.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

    # 6. Save ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], "--", label="Random classifier")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{experiment_name} - ROC Curve")
    ax.legend()
    ax.grid(alpha=0.3)

    fig.savefig(
        output_dir / "roc_curve.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

    # 7. Save training history plot
    if history is not None:

        h = (
            history.history
            if hasattr(history, "history")
            else history
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        for ax, metric in zip(axes, ["loss", "auc"]):
            ax.plot(h[metric], label="Training")
            ax.plot(h[f"val_{metric}"], label="Validation")
            ax.set_title(metric.upper())
            ax.set_xlabel("Epoch")
            ax.legend()
            ax.grid(alpha=0.3)

        fig.tight_layout()

        fig.savefig(
            output_dir / "training_history.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.close(fig)

    print(f"Results saved in: {output_dir}")



## 10. Run the Pipeline

You can run the following cells **step by step**, or use the **end-to-end** cell.

> If your CSV isn’t called `tess_data.csv`, change `CSV_PATH` below.


## 10.0 Data Preparation

In [14]:
# Path to your dataset
CSV_PATH = "../data/tess_data.csv"   # <- change me if needed
N_BINS = 1000                # number of flux bins/columns per sample


In [15]:

np.random.seed(42)
#  1) Load and prepare data
X_train, X_test, y_train, y_test, metadata_test, X_test_orig, X_err_test, scaler = load_data(
    csv_path=CSV_PATH, n_bins=N_BINS
)


LOADING DATA
Dataset: 944 samples

Original distribution:
  Class 0: 472, Class 1: 472
  Ratio: 1.00:1

Initial split - Train: 755, Test: 189

CREATING BALANCED DATASET
Original - Class 0: 377, Class 1: 378
Balanced - Class 0: 350, Class 1: 350

STANDARDIZATION
Train: mean=-0.000000, std=1.000000
Test:  mean=-0.001916, std=0.463545

Final - X_train: (700, 1000, 1), X_test: (189, 1000, 1)
Train dist: 0=350, 1=350


In [16]:
# Reshape for 2D CNN architectures

X_train_2d = X_train.reshape(-1, 50, 20, 1)
X_test_2d = X_test.reshape(-1, 50, 20, 1)

print("CNN 1D:", X_train.shape)
print("CNN 2D:", X_train_2d.shape)

print("Test 1D:", X_test.shape)
print("Test 2D:", X_test_2d.shape)

CNN 1D: (700, 1000, 1)
CNN 2D: (700, 50, 20, 1)
Test 1D: (189, 1000, 1)
Test 2D: (189, 50, 20, 1)


## 10.1 CNN 1 1D

In [17]:
# Puliamo lo stato dei modelli precedenti
tf.keras.backend.clear_session()

# Fissiamo il seed per l'inizializzazione dei pesi
tf.keras.utils.set_random_seed(42)

# 2) Build model
model = build_simple_cnn(n_bins=N_BINS)


BUILDING SIMPLIFIED CNN


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1000, 64)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1000, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 500, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 500, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 500, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 500, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 250, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 250, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,249 (1.61 MB)

 Trainable params: 420,353 (1.60 MB)

 Non-trainable params: 896 (3.50 KB)


Using Focal Loss (gamma=2.5, alpha=0.75)


In [19]:
# 3) Train
history = train_model(model, X_train, y_train, X_test, y_test, epochs=200)


TRAINING
Epoch 1/200
 1/22 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.5312 - auc: 0.5223 - loss: 0.9122 - precision: 0.4000 - recall: 0.3077

W0000 00:00:1795259553.450799  345536 cpu_allocator_impl.cc:82] Allocation of 11904000 exceeds 10% of free system memory.
W0000 00:00:1795259553.450799  345539 cpu_allocator_impl.cc:82] Allocation of 11904000 exceeds 10% of free system memory.
W0000 00:00:1795259553.535705  345539 cpu_allocator_impl.cc:82] Allocation of 16512000 exceeds 10% of free system memory.
W0000 00:00:1795259553.535693  345540 cpu_allocator_impl.cc:82] Allocation of 16512000 exceeds 10% of free system memory.


 2/22 ━━━━━━━━━━━━━━━━━━━━ 4s 216ms/step - accuracy: 0.5234 - auc: 0.5254 - loss: 0.9113 - precision: 0.4344 - recall: 0.4125

W0000 00:00:1795259553.721562  345538 cpu_allocator_impl.cc:82] Allocation of 11904000 exceeds 10% of free system memory.


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.5011 - auc: 0.4939 - loss: 0.8738 - precision: 0.4840 - recall: 0.7960
Epoch 1: val_auc improved from None to 0.54782, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 118ms/step - accuracy: 0.5143 - auc: 0.5065 - loss: 0.8397 - precision: 0.5080 - recall: 0.9029 - val_accuracy: 0.5185 - val_auc: 0.5478 - val_loss: 0.7684 - val_precision: 0.5081 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5047 - auc: 0.4965 - loss: 0.7460 - precision: 0.4921 - recall: 0.9653
Epoch 2: val_auc did not improve from 0.54782
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.5300 - auc: 0.5402 - loss: 0.7202 - precision: 0.5162 - recall: 0.9571 - val_accuracy: 0.5079 - val_auc: 0.5334 - val_loss: 0.6636 - val_precision: 0.5027 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/200
21/22 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.5162 - auc: 0.5142 - l

In [20]:
# 4) Evaluate with optimal threshold
y_pred, y_pred_proba, threshold = evaluate_with_optimal_threshold(model, X_test, y_test)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.2599 (default=0.5)
  At this threshold: TPR=0.9043, FPR=0.1579

Results:
  AUC-ROC: 0.9103
  Accuracy (default threshold=0.5): 0.5979 (59.79%)
  Accuracy (optimal threshold=0.2599): 0.8730 (87.30%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.8989    0.8421    0.8696        95
      Planet     0.8500    0.9043    0.8763        94

    accuracy                         0.8730       189
   macro avg     0.8744    0.8732    0.8729       189
weighted avg     0.8746    0.8730    0.8729       189


Prediction distribution (optimal threshold):
  Predicted 0: 89
  Predicted 1: 100
True distribution:
  True 0: 95
  True 1: 94


In [21]:

# 4. Save results - CNN 1 Focal Loss

save_experiment_results(
    experiment_name="cnn1_focal",
    model=model,
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_pred_proba,
    threshold=threshold,
    history=history
)


Results saved in: taskF_outputs/cnn1_focal


In [ ]:
# 5) Visualize & save artifacts
#plot_all(y_test, y_pred, y_pred_proba, history, metadata_test, X_test, threshold,
#         X_test_orig=X_test_orig, X_err_test=X_err_test, scaler=scaler)

# Persist model and threshold
#model.save('tess_model_final.keras')
#np.save('optimal_threshold.npy', threshold)

#print("\n" + "="*70)
#print("TRAINING COMPLETE!")
#print("="*70)
#print("\nKey improvements:")
#print("  ✓ Perfectly balanced training data")
#print("  ✓ Focal loss for hard examples")
#print("  ✓ Optimal threshold selection")
#print("  ✓ AUC-focused optimization")
#print("\nFiles:")
#print("  - tess_model_final.keras")
#print("  - best_model_final.keras")
#print("  - optimal_threshold.npy")
#print("  - confusion_matrix_final.png")
#print("  - training_history_final.png")
#print("  - sample_lightcurves_predictions.png")
#print("="*70)


VISUALIZATIONS
Saved: confusion_matrix_final.png
Saved: training_history_final.png

PLOTTING LIGHTCURVES WITH PREDICTIONS (n=6)
Saved: sample_lightcurves_predictions.png

TRAINING COMPLETE!

Key improvements:
  ✓ Perfectly balanced training data
  ✓ Focal loss for hard examples
  ✓ Optimal threshold selection
  ✓ AUC-focused optimization

Files:
  - tess_model_final.keras
  - best_model_final.keras
  - optimal_threshold.npy
  - confusion_matrix_final.png
  - training_history_final.png
  - sample_lightcurves_predictions.png


In [25]:

import tensorflow as tf

# Ripartiamo con una nuova inizializzazione
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

# Costruiamo la stessa architettura del professore
model_bce = build_simple_cnn(n_bins=N_BINS)

# Cambiamo SOLO la funzione di perdita
model_bce.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

print("Loss:", model_bce.loss)
print("Parameters:", model_bce.count_params())


BUILDING SIMPLIFIED CNN


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1000, 64)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1000, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 500, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 500, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 500, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 500, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 250, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 250, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,249 (1.61 MB)

 Trainable params: 420,353 (1.60 MB)

 Non-trainable params: 896 (3.50 KB)


Using Focal Loss (gamma=2.5, alpha=0.75)
Loss: binary_crossentropy
Parameters: 421249


In [26]:

# CNN 1B - Training con Binary Cross-Entropy

history_bce = train_model(model_bce, X_train, y_train, X_test, y_test, epochs=200)



TRAINING
Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5195 - auc: 0.5756 - loss: 1.5242 - precision: 0.5212 - recall: 0.1230
Epoch 1: val_auc improved from None to 0.54390, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.5000 - auc: 0.5500 - loss: 1.5141 - precision: 0.5000 - recall: 0.2371 - val_accuracy: 0.5397 - val_auc: 0.5439 - val_loss: 1.4796 - val_precision: 0.5193 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.5410 - auc: 0.5336 - loss: 1.4600 - precision: 0.5234 - recall: 0.7659
Epoch 2: val_auc improved from 0.54390 to 0.65711, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.5843 - auc: 0.5872 - loss: 1.4387 - precision: 0.5758 - recall: 0.6400 - val_accuracy: 0.5397 - val_auc: 0.6571 - val_loss: 1.4284 - val_precision: 0.5193 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/200
22/22 ━━━━━

In [28]:
# Valutazione della CNN con Binary Cross-Entropy

y_pred_bce, y_pred_proba_bce, threshold_bce = (
    evaluate_with_optimal_threshold(
        model_bce,
        X_test,
        y_test
    )
)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.3471 (default=0.5)
  At this threshold: TPR=0.8617, FPR=0.2000

Results:
  AUC-ROC: 0.8686
  Accuracy (default threshold=0.5): 0.5026 (50.26%)
  Accuracy (optimal threshold=0.3471): 0.8307 (83.07%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.8539    0.8000    0.8261        95
      Planet     0.8100    0.8617    0.8351        94

    accuracy                         0.8307       189
   macro avg     0.8320    0.8309    0.8306       189
weighted avg     0.8321    0.8307    0.8305       189


Prediction distribution (optimal threshold):
  Predicted 0: 89
  Predicted 1: 100
True distribution:
  True 0: 95
  True 1: 94


In [29]:

# 4. Save results - CNN 1 Binary Cross-Entropy

save_experiment_results(
    experiment_name="cnn1_bce",
    model=model_bce,
    y_true=y_test,
    y_pred=y_pred_bce,
    y_prob=y_pred_proba_bce,
    threshold=threshold_bce,
    history=history_bce
)


Results saved in: taskF_outputs/cnn1_bce


## 10.1 CNN 2 2D

In [34]:
# 1. Initialize and build model
import tensorflow as tf
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model_cnn2_focal = build_cnn2_2d(loss_type="focal")

print("Parameters:", model_cnn2_focal.count_params())
print("Input:", model_cnn2_focal.input_shape)
print("Output:", model_cnn2_focal.output_shape)

assert model_cnn2_focal.count_params() == 667393

Parameters: 667393
Input: (None, 50, 20, 1)
Output: (None, 1)


In [35]:
# 2. Train model

history_cnn2_focal = train_model(model_cnn2_focal, X_train_2d, y_train, X_test_2d, y_test, epochs=200)


TRAINING
Epoch 1/200
21/22 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4912 - auc: 0.4914 - loss: 0.8924 - precision: 0.4733 - recall: 0.7566
Epoch 1: val_auc improved from None to 0.60162, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - accuracy: 0.5129 - auc: 0.5002 - loss: 0.8640 - precision: 0.5073 - recall: 0.8914 - val_accuracy: 0.5344 - val_auc: 0.6016 - val_loss: 0.8067 - val_precision: 0.5165 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5448 - auc: 0.5626 - loss: 0.7905 - precision: 0.5148 - recall: 0.9427
Epoch 2: val_auc improved from 0.60162 to 0.79703, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.5614 - auc: 0.6030 - loss: 0.7707 - precision: 0.5353 - recall: 0.9314 - val_accuracy: 0.4974 - val_auc: 0.7970 - val_loss: 0.7274 - val_precision: 0.4974 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/200
21/22 ━━━━━

In [36]:
y_pred_cnn2_focal, y_prob_cnn2_focal, threshold_cnn2_focal = (
    evaluate_with_optimal_threshold(model_cnn2_focal, X_test_2d, y_test)
)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.7580 (default=0.5)
  At this threshold: TPR=0.9255, FPR=0.1368

Results:
  AUC-ROC: 0.9127
  Accuracy (default threshold=0.5): 0.5979 (59.79%)
  Accuracy (optimal threshold=0.7580): 0.8942 (89.42%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.9213    0.8632    0.8913        95
      Planet     0.8700    0.9255    0.8969        94

    accuracy                         0.8942       189
   macro avg     0.8957    0.8943    0.8941       189
weighted avg     0.8958    0.8942    0.8941       189


Prediction distribution (optimal threshold):
  Predicted 0: 89
  Predicted 1: 100
True distribution:
  True 0: 95
  True 1: 94


In [37]:

# 4. Save results - CNN 2 Focal Loss

save_experiment_results(
    experiment_name="cnn2_focal",
    model=model_cnn2_focal,
    y_true=y_test,
    y_pred=y_pred_cnn2_focal,
    y_prob=y_prob_cnn2_focal,
    threshold=threshold_cnn2_focal,
    history=history_cnn2_focal
)


Results saved in: taskF_outputs/cnn2_focal


In [38]:
import tensorflow as tf

# 1. Initialize and build model
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model_cnn2_bce = build_cnn2_2d(loss_type="bce")

print("Parameters:", model_cnn2_bce.count_params())
print("Input:", model_cnn2_bce.input_shape)
print("Output:", model_cnn2_bce.output_shape)
print("Loss:", model_cnn2_bce.loss)

assert model_cnn2_bce.count_params() == 667393
assert model_cnn2_bce.loss == "binary_crossentropy"


Parameters: 667393
Input: (None, 50, 20, 1)
Output: (None, 1)
Loss: binary_crossentropy


In [39]:
# 2. Train model

history_cnn2_bce = train_model(model_cnn2_bce, X_train_2d, y_train, X_test_2d, y_test, epochs=200)


TRAINING
Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5273 - auc: 0.5881 - loss: 1.5408 - precision: 0.5266 - recall: 0.1653
Epoch 1: val_auc improved from None to 0.69339, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.5171 - auc: 0.5592 - loss: 1.5349 - precision: 0.5265 - recall: 0.3400 - val_accuracy: 0.5397 - val_auc: 0.6934 - val_loss: 1.5002 - val_precision: 0.5193 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.5888 - auc: 0.6090 - loss: 1.4829 - precision: 0.5748 - recall: 0.6631
Epoch 2: val_auc improved from 0.69339 to 0.72413, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.6300 - auc: 0.6533 - loss: 1.4612 - precision: 0.6409 - recall: 0.5914 - val_accuracy: 0.6402 - val_auc: 0.7241 - val_loss: 1.4637 - val_precision: 0.7500 - val_recall: 0.4149 - learning_rate: 5.0000e-04
Epoch 3/200
21/22 ━━━━

In [40]:

# CNN 2D - Valutazione Binary Cross-Entropy

y_pred_cnn2_bce, y_prob_cnn2_bce, threshold_cnn2_bce = (
    evaluate_with_optimal_threshold(model_cnn2_bce, X_test_2d, y_test)
)



THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.9787 (default=0.5)
  At this threshold: TPR=0.8723, FPR=0.1474

Results:
  AUC-ROC: 0.8950
  Accuracy (default threshold=0.5): 0.5926 (59.26%)
  Accuracy (optimal threshold=0.9787): 0.8624 (86.24%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.8710    0.8526    0.8617        95
      Planet     0.8542    0.8723    0.8632        94

    accuracy                         0.8624       189
   macro avg     0.8626    0.8625    0.8624       189
weighted avg     0.8626    0.8624    0.8624       189


Prediction distribution (optimal threshold):
  Predicted 0: 93
  Predicted 1: 96
True distribution:
  True 0: 95
  True 1: 94


In [41]:

# 4. Save results - CNN 2 Binary Cross-Entropy

save_experiment_results(
    experiment_name="cnn2_bce",
    model=model_cnn2_bce,
    y_true=y_test,
    y_pred=y_pred_cnn2_bce,
    y_prob=y_prob_cnn2_bce,
    threshold=threshold_cnn2_bce,
    history=history_cnn2_bce
)


Results saved in: taskF_outputs/cnn2_bce


## 10.3 CNN 3 1D

Add by me lest try 

In [ ]:
# CNN 3 - Focal Loss compilation
 
# Build architecture (not compiled yet)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model_cnn3 = build_cnn3_1d(n_bins=N_BINS)

model_cnn3.summary()

model_cnn3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss=model.loss,  # Same Focal Loss as CNN 1
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

print("Loss:", model_cnn3.loss)
print("Parameters:", model_cnn3.count_params())
print("Optimizer:", type(model_cnn3.optimizer).__name__)

Loss: <function focal_loss.<locals>.focal_loss_fixed at 0xfffef06a4af0>
Parameters: 243137
Optimizer: Adam


In [44]:
# 2. Train model - CNN 3 Focal Loss

history_cnn3_focal = train_model(
    model_cnn3,
    X_train, y_train,
    X_test, y_test,
    epochs=200
)


TRAINING
Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.5320 - auc: 0.5084 - loss: 0.1344 - precision: 0.5519 - recall: 0.5859
Epoch 1: val_auc improved from None to 0.52032, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 94ms/step - accuracy: 0.5300 - auc: 0.5325 - loss: 0.1265 - precision: 0.5188 - recall: 0.8286 - val_accuracy: 0.5344 - val_auc: 0.5203 - val_loss: 0.1218 - val_precision: 0.5165 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.4975 - auc: 0.5276 - loss: 0.1186 - precision: 0.4883 - recall: 0.9929
Epoch 2: val_auc improved from 0.52032 to 0.54737, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - accuracy: 0.5243 - auc: 0.5481 - loss: 0.1170 - precision: 0.5126 - recall: 0.9857 - val_accuracy: 0.5397 - val_auc: 0.5474 - val_loss: 0.1188 - val_precision: 0.5193 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/200
22/22 ━━━━━

In [45]:
# 3. Evaluate model - CNN 3 Focal Loss

y_pred_cnn3_focal, y_prob_cnn3_focal, threshold_cnn3_focal = (
    evaluate_with_optimal_threshold(model_cnn3, X_test, y_test)
)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.0832 (default=0.5)
  At this threshold: TPR=0.9362, FPR=0.2000

Results:
  AUC-ROC: 0.9101
  Accuracy (default threshold=0.5): 0.5185 (51.85%)
  Accuracy (optimal threshold=0.0832): 0.8677 (86.77%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.9268    0.8000    0.8588        95
      Planet     0.8224    0.9362    0.8756        94

    accuracy                         0.8677       189
   macro avg     0.8746    0.8681    0.8672       189
weighted avg     0.8749    0.8677    0.8671       189


Prediction distribution (optimal threshold):
  Predicted 0: 82
  Predicted 1: 107
True distribution:
  True 0: 95
  True 1: 94


In [ ]:
# 4. Save results - CNN 3 Focal Loss

save_experiment_results(
    experiment_name="cnn3_focal",
    model=model_cnn3,
    y_true=y_test,
    y_pred=y_pred_cnn3_focal,
    y_prob=y_prob_cnn3_focal,
    threshold=threshold_cnn3_focal,
    history=history_cnn3_focal
)

Results saved in: taskF_outputs/cnn3_focal


In [48]:
# 1. Initialize and compile model - CNN 3 BCE

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model_cnn3_bce = build_cnn3_1d(n_bins=N_BINS)

model_cnn3_bce.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

print("Loss:", model_cnn3_bce.loss)
print("Parameters:", model_cnn3_bce.count_params())
print("Optimizer:", type(model_cnn3_bce.optimizer).__name__)

Loss: binary_crossentropy
Parameters: 243137
Optimizer: Adam


In [49]:
# 2. Train model - CNN 3 BCE

history_cnn3_bce = train_model(
    model_cnn3_bce,
    X_train, y_train,
    X_test, y_test,
    epochs=200
)


TRAINING
Epoch 1/200
21/22 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.5362 - auc: 0.5025 - loss: 0.7648 - precision: 0.6302 - recall: 0.1005
Epoch 1: val_auc improved from None to 0.57368, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.5171 - auc: 0.5325 - loss: 0.7588 - precision: 0.5349 - recall: 0.2629 - val_accuracy: 0.5450 - val_auc: 0.5737 - val_loss: 0.7558 - val_precision: 0.5222 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.5230 - auc: 0.5392 - loss: 0.7492 - precision: 0.5058 - recall: 0.7375
Epoch 2: val_auc improved from 0.57368 to 0.70134, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 113ms/step - accuracy: 0.5614 - auc: 0.5768 - loss: 0.7418 - precision: 0.5485 - recall: 0.6943 - val_accuracy: 0.5026 - val_auc: 0.7013 - val_loss: 0.7535 - val_precision: 0.5000 - val_recall: 0.0106 - learning_rate: 5.0000e-04
Epoch 3/200
22/22 ━━━━

In [50]:
# 3. Evaluate model - CNN 3 BCE

y_pred_cnn3_bce, y_prob_cnn3_bce, threshold_cnn3_bce = (
    evaluate_with_optimal_threshold(
        model_cnn3_bce,
        X_test,
        y_test
    )
)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.9018 (default=0.5)
  At this threshold: TPR=0.9362, FPR=0.1474

Results:
  AUC-ROC: 0.9142
  Accuracy (default threshold=0.5): 0.5661 (56.61%)
  Accuracy (optimal threshold=0.9018): 0.8942 (89.42%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.9310    0.8526    0.8901        95
      Planet     0.8627    0.9362    0.8980        94

    accuracy                         0.8942       189
   macro avg     0.8969    0.8944    0.8940       189
weighted avg     0.8971    0.8942    0.8940       189


Prediction distribution (optimal threshold):
  Predicted 0: 87
  Predicted 1: 102
True distribution:
  True 0: 95
  True 1: 94


In [51]:
# 4. Save results - CNN 3 BCE

save_experiment_results(
    experiment_name="cnn3_bce",
    model=model_cnn3_bce,
    y_true=y_test,
    y_pred=y_pred_cnn3_bce,
    y_prob=y_prob_cnn3_bce,
    threshold=threshold_cnn3_bce,
    history=history_cnn3_bce
)

Results saved in: taskF_outputs/cnn3_bce


## 10.4 CNN 3 1D

In [55]:
# Initialize model with reproducible seed
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model_cnn4 = build_cnn4_2d(
    input_shape=(50, 20, 1)
)

model_cnn4.summary()

# 1. Compile model - CNN 4 Focal Loss

model_cnn4.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss=model.loss,  # Same Focal Loss used in CNN 1
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

print("Loss:", model_cnn4.loss)
print("Parameters:", model_cnn4.count_params())
print("Optimizer:", type(model_cnn4.optimizer).__name__)

Model: "CNN4_Compact_2D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50, 20, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 50, 20, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 25, 10, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 25, 10, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 25, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 5, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 5, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,097 (391.00 KB)

 Trainable params: 99,713 (389.50 KB)

 Non-trainable params: 384 (1.50 KB)

Loss: <function focal_loss.<locals>.focal_loss_fixed at 0xfffef06a4af0>
Parameters: 100097
Optimizer: Adam


In [56]:
# 2. Train model - CNN 4 Focal Loss

history_cnn4_focal = train_model(
    model_cnn4,
    X_train_2d, y_train,
    X_test_2d, y_test,
    epochs=200
)


TRAINING
Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5199 - auc: 0.5561 - loss: 0.0995 - precision: 0.5000 - recall: 0.9200
Epoch 1: val_auc improved from None to 0.51943, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 96ms/step - accuracy: 0.5314 - auc: 0.5524 - loss: 0.0931 - precision: 0.5174 - recall: 0.9343 - val_accuracy: 0.5503 - val_auc: 0.5194 - val_loss: 0.0839 - val_precision: 0.5251 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.5027 - auc: 0.5487 - loss: 0.0921 - precision: 0.4908 - recall: 0.9774
Epoch 2: val_auc improved from 0.51943 to 0.55515, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.5200 - auc: 0.5178 - loss: 0.0906 - precision: 0.5105 - recall: 0.9686 - val_accuracy: 0.5450 - val_auc: 0.5552 - val_loss: 0.0804 - val_precision: 0.5222 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/200
22/22 ━━━━

In [57]:
# 3. Evaluate model - CNN 4 Focal Loss

y_pred_cnn4_focal, y_prob_cnn4_focal, threshold_cnn4_focal = (
    evaluate_with_optimal_threshold(
        model_cnn4,
        X_test_2d,
        y_test
    )
)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.5340 (default=0.5)
  At this threshold: TPR=0.9149, FPR=0.1789

Results:
  AUC-ROC: 0.8944
  Accuracy (default threshold=0.5): 0.5503 (55.03%)
  Accuracy (optimal threshold=0.5340): 0.8677 (86.77%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.9070    0.8211    0.8619        95
      Planet     0.8350    0.9149    0.8731        94

    accuracy                         0.8677       189
   macro avg     0.8710    0.8680    0.8675       189
weighted avg     0.8712    0.8677    0.8675       189


Prediction distribution (optimal threshold):
  Predicted 0: 86
  Predicted 1: 103
True distribution:
  True 0: 95
  True 1: 94


In [58]:
# 4. Save results - CNN 4 Focal Loss

save_experiment_results(
    experiment_name="cnn4_focal",
    model=model_cnn4,
    y_true=y_test,
    y_pred=y_pred_cnn4_focal,
    y_prob=y_prob_cnn4_focal,
    threshold=threshold_cnn4_focal,
    history=history_cnn4_focal
)

Results saved in: taskF_outputs/cnn4_focal


In [59]:

# 1. Initialize and compile model - CNN 4 BCE

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model_cnn4_bce = build_cnn4_2d(
    input_shape=(50, 20, 1)
)

model_cnn4_bce.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

print("Loss:", model_cnn4_bce.loss)
print("Parameters:", model_cnn4_bce.count_params())
print("Optimizer:", type(model_cnn4_bce.optimizer).__name__)


Loss: binary_crossentropy
Parameters: 100097
Optimizer: Adam


In [60]:
 # 2. Train model - CNN 4 BCE

history_cnn4_bce = train_model(
    model_cnn4_bce,
    X_train_2d, y_train,
    X_test_2d, y_test,
    epochs=200
)


TRAINING
Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.5228 - auc: 0.5255 - loss: 0.7589 - precision: 0.5042 - recall: 0.7486
Epoch 1: val_auc improved from None to 0.60974, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 161ms/step - accuracy: 0.5257 - auc: 0.5200 - loss: 0.7352 - precision: 0.5196 - recall: 0.6829 - val_accuracy: 0.5450 - val_auc: 0.6097 - val_loss: 0.7092 - val_precision: 0.5225 - val_recall: 0.9894 - learning_rate: 5.0000e-04
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.5308 - auc: 0.5892 - loss: 0.7294 - precision: 0.5116 - recall: 0.6627
Epoch 2: val_auc improved from 0.60974 to 0.74759, saving model to best_model_final.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step - accuracy: 0.5343 - auc: 0.5537 - loss: 0.7306 - precision: 0.5324 - recall: 0.5629 - val_accuracy: 0.5238 - val_auc: 0.7476 - val_loss: 0.7054 - val_precision: 0.8333 - val_recall: 0.0532 - learning_rate: 5.0000e-04
Epoch 3/200
22/22 ━━━

In [61]:
# 3. Evaluate model - CNN 4 BCE

y_pred_cnn4_bce, y_prob_cnn4_bce, threshold_cnn4_bce = (
    evaluate_with_optimal_threshold(
        model_cnn4_bce,
        X_test_2d,
        y_test
    )
)


THRESHOLD OPTIMIZATION & EVALUATION

Optimal threshold: 0.3143 (default=0.5)
  At this threshold: TPR=0.9468, FPR=0.2105

Results:
  AUC-ROC: 0.9029
  Accuracy (default threshold=0.5): 0.7249 (72.49%)
  Accuracy (optimal threshold=0.3143): 0.8677 (86.77%)

With optimal threshold:
              precision    recall  f1-score   support

  Non-Planet     0.9375    0.7895    0.8571        95
      Planet     0.8165    0.9468    0.8768        94

    accuracy                         0.8677       189
   macro avg     0.8770    0.8681    0.8670       189
weighted avg     0.8773    0.8677    0.8669       189


Prediction distribution (optimal threshold):
  Predicted 0: 80
  Predicted 1: 109
True distribution:
  True 0: 95
  True 1: 94


In [62]:
# 4. Save results - CNN 4 BCE

save_experiment_results(
    experiment_name="cnn4_bce",
    model=model_cnn4_bce,
    y_true=y_test,
    y_pred=y_pred_cnn4_bce,
    y_prob=y_prob_cnn4_bce,
    threshold=threshold_cnn4_bce,
    history=history_cnn4_bce
)

Results saved in: taskF_outputs/cnn4_bce


## 11. (Optional) One-Click: Run Everything

This cell wraps all steps into a single function for convenience.


In [15]:
def main():
    X_train, X_test, y_train, y_test, metadata_test, X_test_orig, X_err_test, scaler = load_data(
        csv_path=CSV_PATH, n_bins=N_BINS
    )
    model = build_simple_cnn(n_bins=N_BINS)
    history = train_model(model, X_train, y_train, X_test, y_test, epochs=200)
    y_pred, y_pred_proba, threshold = evaluate_with_optimal_threshold(model, X_test, y_test)
    plot_all(y_test, y_pred, y_pred_proba, history, metadata_test, X_test, threshold,
             X_test_orig=X_test_orig, X_err_test=X_err_test, scaler=scaler)
    model.save('tess_model_final.keras')
    np.save('optimal_threshold.npy', threshold)
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)
    print("\nKey improvements:")
    print("  ✓ Perfectly balanced training data")
    print("  ✓ Focal loss for hard examples")
    print("  ✓ Optimal threshold selection")
    print("  ✓ AUC-focused optimization")
    print("\nFiles:")
    print("  - tess_model_final.keras")
    print("  - best_model_final.keras")
    print("  - optimal_threshold.npy")
    print("  - confusion_matrix_final.png")
    print("  - training_history_final.png")
    print("  - sample_lightcurves_predictions.png")
    print("="*70)

# Uncomment to run end-to-end:
# main()

## 12. Interpreting Results & Next Steps

- **AUC-ROC** is the primary score during training. Inspect training curves to ensure you’re not overfitting.  
- **Confusion matrix** with counts and percentages helps quantify trade-offs at the **optimal threshold**.  
- **False positives** vs **false negatives**: use domain needs to decide how to tune `alpha`/`gamma` in focal loss or to move the threshold.

**Ideas to try next**

- Add **class-dependent augmentations** (e.g., transit-like dips for positives).  
- Calibrate probabilities (e.g., **Platt scaling**, **isotonic regression**) for better decision thresholds.  
- Incorporate additional channels (centroid motion, background, etc.) into a **multi-input** model.  
- Use **cross-validation** on the training set to measure variability across folds.


---

### Appendix: Notes on Data Schema

- Ensure your CSV contains **exactly** `n_bins` columns named `flux_0000 .. flux_{n_bins-1:04d}` and matching `flux_err_*` columns.
- Metadata columns are optional for training but used for prettier plots.

### Troubleshooting

- `ValueError: columns not found`: your CSV headers don’t match the expected names. Check `n_bins` and column prefixes.  
- `CUDA out of memory`: reduce `batch_size`, or limit GPU memory; try the provided GPU memory-growth snippet.  
- `AUC not improving`: try a bigger `samples_per_class`, more dropout, or adjust `gamma`/`alpha`.

---
